# 03 · Model Eğitimi (Model Training)

Eğitim yalnızca **train** bölümünü, model seçimi (early stopping, en iyi checkpoint) yalnızca **val** bölümünü
kullanır. Test bölümü bu notebook'ta hiç yüklenmez.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "passive_radar").is_dir())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from dataclasses import asdict, replace

import matplotlib.pyplot as plt

from passive_radar.config import CLASS_NAMES, ProjectConfig
from passive_radar.data.dataset import RangeDopplerDataset, make_data_loader
from passive_radar.data.indexing import load_split_csv, records_in_split
from passive_radar.data.preprocessing import load_channel_stats
from passive_radar.evaluation.plots import plot_training_history
from passive_radar.models.cnn import RangeDopplerCNN, count_trainable_parameters
from passive_radar.training.runtime import resolve_device, set_seed
from passive_radar.training.trainer import Trainer

## 1. Yapılandırma

In [ ]:
config = ProjectConfig.default(PROJECT_ROOT)

# Experiment knobs: change here (or in config.py) instead of editing library code.
training_config = replace(config.training, batch_size=16, learning_rate=1e-3, max_epochs=30, early_stopping_patience=5)
model_config = config.model

mean, std = load_channel_stats(config.paths.normalization_json)
preprocessing_config = replace(config.preprocessing, mean=mean, std=std)

for name, section in (("preprocessing", preprocessing_config), ("model", model_config), ("training", training_config)):
    print(f"{name}: {asdict(section)}")

## 2. Eğitim ve doğrulama verilerinin yüklenmesi

In [ ]:
records = load_split_csv(config.paths.splits_csv, relative_to=config.paths.project_root)
train_records, val_records = records_in_split(records, "train"), records_in_split(records, "val")

train_set = RangeDopplerDataset(train_records, preprocessing_config, cache_images=training_config.cache_images)
val_set = RangeDopplerDataset(val_records, preprocessing_config, cache_images=training_config.cache_images)
train_loader = make_data_loader(train_set, training_config.batch_size, shuffle=True,
                                num_workers=training_config.num_workers, seed=training_config.seed)
val_loader = make_data_loader(val_set, training_config.batch_size, shuffle=False,
                              num_workers=training_config.num_workers)
print(f"train: {len(train_set)} images | val: {len(val_set)} images")

## 3. CNN modelinin oluşturulması

In [ ]:
set_seed(training_config.seed)
device = resolve_device(training_config.device)
model = RangeDopplerCNN(model_config)
print("device:", device, "| trainable parameters:", f"{count_trainable_parameters(model):,}")
model

## 4. Eğitim döngüsü (early stopping + en iyi modelin kaydı)

In [ ]:
trainer = Trainer(model, training_config, device, config.paths.checkpoint_path, CLASS_NAMES, preprocessing_config)
history = trainer.fit(train_loader, val_loader)
history.save_json(config.paths.outputs_dir / "training_history.json")
print("best epoch:", history.best_epoch, "| stopped early:", history.stopped_early)
print("checkpoint:", config.paths.checkpoint_path.relative_to(config.paths.project_root))

## 5. Eğitim ve doğrulama kayıpları

In [ ]:
plot_training_history(history, save_path=config.paths.outputs_dir / "figures" / "training_history.png")
plt.show()

En iyi model **en düşük doğrulama kaybına** göre seçilir ve `models/best_model.pt` dosyasına; ağırlıklar, sınıf
isimleri, model yapılandırması ve ön işleme yapılandırması (normalizasyon dahil) birlikte kaydedilir. Bağımsız
değerlendirme `04_evaluation.ipynb` içinde yapılır.